# bagging, dbscan, adaboost

In [1]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, silhouette_score

In [2]:
# bagging on iris
iris_paths = [Path('Iris.csv'), Path('iris.csv'), Path('Lab 11/Iris.csv'), Path('Lab 11/iris.csv')]
iris_path = next((p for p in iris_paths if p.exists()), None)
if iris_path is None:
    from sklearn.datasets import load_iris
    iris_frame = load_iris(as_frame=True)
    iris_df = iris_frame.frame
else:
    iris_df = pd.read_csv(iris_path)
if 'Species' in iris_df.columns:
    y_iris = iris_df['Species']
    drop_cols = [c for c in ['Species', 'Id'] if c in iris_df.columns]
    X_iris = iris_df.drop(columns=drop_cols)
else:
    target_col = iris_df.columns[-1]
    y_iris = iris_df[target_col]
    X_iris = iris_df.drop(columns=[target_col])
X_train, X_temp, y_train, y_temp = train_test_split(X_iris, y_iris, test_size=0.4, stratify=y_iris, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
bag_model = BaggingClassifier(DecisionTreeClassifier(random_state=42), n_estimators=200, random_state=42)
bag_model.fit(X_train, y_train)
val_pred = bag_model.predict(X_val)
test_pred = bag_model.predict(X_test)
print('bagging val accuracy:', round(accuracy_score(y_val, val_pred), 4))
print('bagging test accuracy:', round(accuracy_score(y_test, test_pred), 4))

bagging val accuracy: 0.9333
bagging test accuracy: 0.9667


In [3]:
# dbscan on iris
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)
dbscan_model = DBSCAN(eps=0.6, min_samples=5)
cluster_labels = dbscan_model.fit_predict(X_scaled)
unique_clusters = [c for c in np.unique(cluster_labels) if c != -1]
cluster_map = {}
for c in unique_clusters:
    species = pd.Series(y_iris)[cluster_labels == c]
    if not species.empty:
        cluster_map[c] = species.mode().iloc[0]
mapped_labels = np.array(['noise'] * len(cluster_labels), dtype=object)
for idx, c in enumerate(cluster_labels):
    if c in cluster_map:
        mapped_labels[idx] = cluster_map[c]
mask = cluster_labels != -1
if mask.any():
    dbscan_acc = accuracy_score(np.array(y_iris)[mask], mapped_labels[mask])
else:
    dbscan_acc = float('nan')
if len(unique_clusters) > 1 and mask.sum() > len(unique_clusters):
    sil_score = silhouette_score(X_scaled[mask], cluster_labels[mask])
else:
    sil_score = float('nan')
print('dbscan usable points:', int(mask.sum()))
print('dbscan accuracy:', round(dbscan_acc, 4))
print('dbscan silhouette:', round(sil_score, 4) if not np.isnan(sil_score) else 'nan')

dbscan usable points: 124
dbscan accuracy: 0.7097
dbscan silhouette: 0.6405


In [ ]:
# adaboost on credit score
credit_paths = [Path('credit_score.csv'), Path('Score.csv'), Path('Lab 11/Score.csv'), Path('Lab 11/Credit_Score.csv')]
credit_path = next((p for p in credit_paths if p.exists()), None)
if credit_path is None:
    raise FileNotFoundError('credit score csv not found in workspace')
credit_df = pd.read_csv(credit_path)
target_col = 'Credit_Score' if 'Credit_Score' in credit_df.columns else credit_df.columns[-1]
y_credit = credit_df[target_col]
X_credit = credit_df.drop(columns=[target_col])
cat_cols = X_credit.select_dtypes(include='object').columns.tolist()
X_credit = pd.get_dummies(X_credit, columns=cat_cols, drop_first=True)
y_credit_enc = LabelEncoder().fit_transform(y_credit)
X_train, X_temp, y_train, y_temp = train_test_split(X_credit, y_credit_enc, test_size=0.2, stratify=y_credit_enc, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
base_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
ada_model = AdaBoostClassifier(estimator=base_tree, n_estimators=300, learning_rate=0.5, algorithm='SAMME', random_state=42)
ada_model.fit(X_train, y_train)
for name, y_true, y_pred in [('validation', y_val, ada_model.predict(X_val)), ('test', y_test, ada_model.predict(X_test))]:
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    print(f'adaboost {name} accuracy:', round(acc, 4))
    print(f'adaboost {name} precision:', round(prec, 4))
    print(f'adaboost {name} recall:', round(rec, 4))
    print(f'adaboost {name} f1:', round(f1, 4))

c:\Users\lenovo\anaconda3\envs\ai-gpu\lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


adaboost validation accuracy: 0.642
adaboost validation precision: 0.6251
adaboost validation recall: 0.5969
adaboost validation f1: 0.6085
adaboost test accuracy: 0.6549
adaboost test precision: 0.6409
adaboost test recall: 0.6128
adaboost test f1: 0.6245
